In [28]:
from google.colab import files

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

print("Libraries imported successfully!")

Libraries imported successfully!


In [30]:
import pandas as pd
from pathlib import Path

# Final cleaned datasets
housing_path = Path("/content/housing_clean .csv")
crash_path = Path("/content/crash_clean.csv")

if not housing_path.exists():
    raise FileNotFoundError(
        "Upload Member 1's final housing_clean.csv to Colab."
    )

if not crash_path.exists():
    raise FileNotFoundError(
        "Upload Member 2's final crash_clean.csv to Colab."
    )

housing = pd.read_csv(
    housing_path,
    low_memory=False
)

transport = pd.read_csv(
    crash_path,
    low_memory=False
)

print("HOUSING DATASET")
print("Shape:", housing.shape)
print("Columns:", housing.columns.tolist())
display(housing.head())

print("\nTRANSPORTATION DATASET")
print("Shape:", transport.shape)
print("Columns:", transport.columns.tolist())
display(transport.head())

print("\nBoth FINAL datasets loaded successfully!")

HOUSING DATASET
Shape: (381142, 24)
Columns: ['job_base', 'job_filing_number', 'filing_date', 'job_type', 'borough', 'zip', 'zip_source', 'latitude', 'longitude', 'proposed_dwelling_units', 'proposed_no_of_stories', 'initial_cost', 'first_permit_date', 'signoff_date', 'days_permit_to_signoff', 'is_sidewalk_shed', 'is_curb_cut', 'year', 'month', 'year_month', 'job_suffix', 'proposed_dwelling_units_iqr_outlier', 'proposed_no_of_stories_iqr_outlier', 'initial_cost_iqr_outlier']


,job_base,job_filing_number,filing_date,job_type,borough,zip,zip_source,latitude,longitude,proposed_dwelling_units,...,days_permit_to_signoff,is_sidewalk_shed,is_curb_cut,year,month,year_month,job_suffix,proposed_dwelling_units_iqr_outlier,proposed_no_of_stories_iqr_outlier,initial_cost_iqr_outlier
0,B00642615,B00642615-I1,2022-01-01 00:00:00,Alteration,Brooklyn,11222.0,original,40.724361,-73.955004,0.0,...,NaN,False,False,2022,1,2022-01,I1,False,False,False
1,Q00655660,Q00655660-I1,2022-01-01 00:00:00,Alteration,Queens,11434.0,original,40.663741,-73.771273,2.0,...,223.0,False,False,2022,1,2022-01,I1,False,False,False
2,M00655760,M00655760-I1,2022-01-01 00:00:00,Alteration,Manhattan,10023.0,original,40.776083,-73.987179,561.0,...,111.0,False,False,2022,1,2022-01,I1,True,True,False
3,Q00655698,Q00655698-I1,2022-01-01 00:00:00,Alteration,Queens,11357.0,original,40.779402,-73.802022,2.0,...,NaN,False,False,2022,1,2022-01,I1,False,False,False
4,Q00650444,Q00650444-I1,2022-01-01 00:00:00,Alteration,Queens,11101.0,original,40.754212,-73.918442,1.0,...,1004.0,False,False,2022,1,2022-01,I1,False,False,False



TRANSPORTATION DATASET
Shape: (13309, 21)
Columns: ['open_data_crash_id', 'crash_date', 'crash_time', 'crash_hour', 'crash_weekday', 'borough', 'borough_source', 'zip', 'zip_source', 'latitude', 'longitude', 'total_severe_injuries', 'total_non_severe_injuries', 'total_fatalities', 'similar_fatalities_severe', 'is_highway', 'is_repeat_location', 'year', 'month', 'year_month', 'time_is_midnight']


,open_data_crash_id,crash_date,crash_time,crash_hour,crash_weekday,borough,borough_source,zip,zip_source,latitude,...,total_severe_injuries,total_non_severe_injuries,total_fatalities,similar_fatalities_severe,is_highway,is_repeat_location,year,month,year_month,time_is_midnight
0,4492364,2022-01-01,01:28,1,Saturday,Bronx,original,10468.0,spatial_join,40.858707,...,1.0,1.0,0.0,0.0,False,False,2022,1,2022-01,False
1,4491159,2022-01-01,04:33,4,Saturday,Bronx,original,NaN,missing,NaN,...,1.0,0.0,0.0,0.0,False,False,2022,1,2022-01,False
2,4491487,2022-01-01,05:55,5,Saturday,Bronx,original,10467.0,spatial_join,40.871550,...,1.0,0.0,0.0,0.0,False,False,2022,1,2022-01,False
3,4491223,2022-01-01,10:42,10,Saturday,Bronx,original,10473.0,spatial_join,40.823673,...,1.0,0.0,0.0,0.0,True,False,2022,1,2022-01,False
4,4491418,2022-01-01,20:55,20,Saturday,Bronx,original,10467.0,spatial_join,40.863945,...,1.0,0.0,0.0,0.0,False,False,2022,1,2022-01,False



Both FINAL datasets loaded successfully!


In [31]:
# Check Member 1's cleaned Housing data
print("HOUSING CLEANING CHECK")

important_columns = [
    "job_base",
    "filing_date",
    "h_filings_total",
    "h_new_building"
]

for col in important_columns:
    if col in housing.columns:
        print(f"{col}: Found")
    else:
        print(f"{col}: Not found")

if "job_base" in housing.columns:
    print(
        "Duplicate job bases:",
        housing["job_base"].duplicated().sum()
    )

HOUSING CLEANING CHECK
job_base: Found
filing_date: Found
h_filings_total: Not found
h_new_building: Not found
Duplicate job bases: 0


## 1. Integration Strategy and Justification

### Objective

The objective of this stage is to integrate NYC housing development records from DOB NOW with transportation safety records from SIRTA.

The integration will help us explore whether areas with higher building activity also show different patterns in transportation safety records.

### Proposed Integration Method

The preferred method is to connect both datasets using ZIP code and month, provided both datasets contain reliable geographic and date information.

Before merging, each dataset will be aggregated separately to avoid multiplying records that share the same ZIP code and month.

### Why This Method?

- ZIP codes allow geographic comparisons across NYC.
- Months allow us to examine changes over time.
- Aggregation reduces unnecessary duplication and makes the datasets easier to compare.
- A merge indicator helps identify which geographic and time combinations appear in both sources.

### Limitations

The results can show associations but cannot establish that construction activity causes transportation safety incidents.

Differences in population, road usage, traffic volume, and reporting procedures may influence the results.


In [32]:
# Work on copies to preserve the original data
housing_df = housing.copy()
transport_df = transport.copy()

# Standardize column names
def standardize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

housing_df = standardize_columns(housing_df)
transport_df = standardize_columns(transport_df)

print("HOUSING COLUMNS")
print(housing_df.columns.tolist())

print("\nSIRTA COLUMNS")
print(transport_df.columns.tolist())

HOUSING COLUMNS
['job_base', 'job_filing_number', 'filing_date', 'job_type', 'borough', 'zip', 'zip_source', 'latitude', 'longitude', 'proposed_dwelling_units', 'proposed_no_of_stories', 'initial_cost', 'first_permit_date', 'signoff_date', 'days_permit_to_signoff', 'is_sidewalk_shed', 'is_curb_cut', 'year', 'month', 'year_month', 'job_suffix', 'proposed_dwelling_units_iqr_outlier', 'proposed_no_of_stories_iqr_outlier', 'initial_cost_iqr_outlier']

SIRTA COLUMNS
['open_data_crash_id', 'crash_date', 'crash_time', 'crash_hour', 'crash_weekday', 'borough', 'borough_source', 'zip', 'zip_source', 'latitude', 'longitude', 'total_severe_injuries', 'total_non_severe_injuries', 'total_fatalities', 'similar_fatalities_severe', 'is_highway', 'is_repeat_location', 'year', 'month', 'year_month', 'time_is_midnight']


In [33]:
# Integration columns confirmed from the cleaned datasets

housing_zip = "zip"
housing_date = "filing_date"
housing_borough = "borough"

transport_zip = "zip"
transport_date = "crash_date"
transport_borough = "borough"

print("HOUSING")
print("ZIP:", housing_zip)
print("Date:", housing_date)
print("Borough:", housing_borough)

print("\nSIRTA")
print("ZIP:", transport_zip)
print("Date:", transport_date)
print("Borough:", transport_borough)

HOUSING
ZIP: zip
Date: filing_date
Borough: borough

SIRTA
ZIP: zip
Date: crash_date
Borough: borough


In [34]:
# Stop if the required columns could not be identified
required = {
    "Housing ZIP": housing_zip,
    "Housing date": housing_date,
    "SIRTA ZIP": transport_zip,
    "SIRTA date": transport_date
}

missing = [
    name for name, value in required.items()
    if value is None
]

if missing:
    raise ValueError(
        "These required columns need manual mapping: "
        + ", ".join(missing)
    )

# Standardize ZIP codes
def clean_zip(series):
    return (
        series.astype("string")
        .str.strip()
        .str.extract(r"^(\d{5})(?:\.0)?$", expand=False)
    )

# Standardize boroughs
def clean_borough(series):
    mapping = {
        "MN": "Manhattan",
        "MANHATTAN": "Manhattan",
        "BK": "Brooklyn",
        "BROOKLYN": "Brooklyn",
        "BX": "Bronx",
        "BRONX": "Bronx",
        "QN": "Queens",
        "QUEENS": "Queens",
        "SI": "Staten Island",
        "STATEN ISLAND": "Staten Island"
    }

    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .map(mapping)
        .astype("string")
    )

# Create standardized integration fields
housing_df["zip_code"] = clean_zip(housing_df[housing_zip])
transport_df["zip_code"] = clean_zip(transport_df[transport_zip])

# Parse mixed date formats correctly
housing_df["date"] = pd.to_datetime(
    housing_df[housing_date],
    format="mixed",
    errors="coerce"
)

transport_df["date"] = pd.to_datetime(
    transport_df[transport_date],
    format="mixed",
    errors="coerce"
)

print("Housing missing dates:", housing_df["date"].isna().sum())
print("SIRTA missing dates:", transport_df["date"].isna().sum())

housing_df["month"] = housing_df["date"].dt.to_period("M")
transport_df["month"] = transport_df["date"].dt.to_period("M")

housing_df["borough"] = (
    clean_borough(housing_df[housing_borough])
    if housing_borough else pd.Series(
        pd.NA, index=housing_df.index, dtype="string"
    )
)

transport_df["borough"] = (
    clean_borough(transport_df[transport_borough])
    if transport_borough else pd.Series(
        pd.NA, index=transport_df.index, dtype="string"
    )
)

# Examine key completeness
for name, df in [
    ("Housing", housing_df),
    ("SIRTA", transport_df)
]:
    print(f"\n{name} integration key quality")
    print("Missing ZIP:", df["zip_code"].isna().sum())
    print("Missing date:", df["date"].isna().sum())
    print("Missing borough:", df["borough"].isna().sum())
    print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nStandardization completed.")

Housing missing dates: 0
SIRTA missing dates: 0

Housing integration key quality
Missing ZIP: 411
Missing date: 0
Missing borough: 0
Date range: 2022-01-01 00:00:00 to 2026-06-30 00:00:00

SIRTA integration key quality
Missing ZIP: 1180
Missing date: 0
Missing borough: 10
Date range: 2022-01-01 00:00:00 to 2026-06-30 00:00:00

Standardization completed.


In [35]:
# Investigate missing Housing filing dates

print("Original missing filing dates:")
print(housing["filing_date"].isna().sum())

print("\nSample original filing dates:")
print(housing["filing_date"].dropna().head(15).tolist())

# Try parsing multiple date formats
dates_mixed = pd.to_datetime(
    housing["filing_date"],
    format="mixed",
    errors="coerce"
)

print("\nMissing dates after mixed-format parsing:")
print(dates_mixed.isna().sum())

print("\nDates successfully recovered compared with current parsing:")
print(
    housing_df["date"].isna().sum()
    - dates_mixed.isna().sum()
)

print("\nExamples of originally unparsed dates:")
display(
    housing.loc[
        housing_df["date"].isna()
        & dates_mixed.notna(),
        ["filing_date"]
    ].head(10)
)

Original missing filing dates:
0

Sample original filing dates:
['2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00', '2022-01-01 00:00:00']

Missing dates after mixed-format parsing:
0

Dates successfully recovered compared with current parsing:
0

Examples of originally unparsed dates:


,filing_date


In [36]:
# Check integration readiness

print("HOUSING")
print("Total records:", len(housing_df))
print("Missing ZIP:", housing_df["zip_code"].isna().sum())
print("Missing dates:", housing_df["date"].isna().sum())
print("Date range:", housing_df["date"].min(),
      "to", housing_df["date"].max())

print("\nSIRTA")
print("Total records:", len(transport_df))
print("Missing ZIP:", transport_df["zip_code"].isna().sum())
print("Missing dates:", transport_df["date"].isna().sum())
print("Date range:", transport_df["date"].min(),
      "to", transport_df["date"].max())

housing_ready = housing_df.dropna(
    subset=["zip_code", "date"]
)

transport_ready = transport_df.dropna(
    subset=["zip_code", "date"]
)

print("\nHousing usable records:", len(housing_ready))
print("SIRTA usable records:", len(transport_ready))

shared_zips = (
    set(housing_ready["zip_code"])
    & set(transport_ready["zip_code"])
)

print("Number of shared ZIP codes:", len(shared_zips))

HOUSING
Total records: 381142
Missing ZIP: 411
Missing dates: 0
Date range: 2022-01-01 00:00:00 to 2026-06-30 00:00:00

SIRTA
Total records: 13309
Missing ZIP: 1180
Missing dates: 0
Date range: 2022-01-01 00:00:00 to 2026-06-30 00:00:00

Housing usable records: 380731
SIRTA usable records: 12129
Number of shared ZIP codes: 191


In [37]:
# Check whether ZIP 10000 exists in NYC's official ZCTA dataset

zcta_url = (
    "https://data.cityofnewyork.us/api/v3/views/"
    "35j5-n34v/export.csv?accessType=DOWNLOAD"
)

zcta = pd.read_csv(zcta_url, dtype=str)

print("ZCTA columns:", zcta.columns.tolist())

zip_column = next(
    (col for col in zcta.columns
     if col.lower().strip() in ["zip code", "zcta5"]),
    None
)

if zip_column is None:
    raise ValueError("Could not identify the ZCTA ZIP column.")

valid_zctas = set(zcta[zip_column].dropna().str.strip())

print("Number of official ZCTAs:", len(valid_zctas))
print("ZIP 10000 found:", "10000" in valid_zctas)

print("\nHousing ZIPs without matching ZCTAs:")
print(
    sorted(set(housing_df["zip_code"].dropna()) - valid_zctas)[:30]
)

print("\nSIRTA ZIPs without matching ZCTAs:")
print(
    sorted(set(transport_df["zip_code"].dropna()) - valid_zctas)[:30]
)

ZCTA columns: ['Zip Code', 'Land Area', 'Water Area', 'Center Latitude', 'Center Longitude', 'Internal Latitude', 'Internal Longitude', 'Geometry']
Number of official ZCTAs: 221
ZIP 10000 found: False

Housing ZIPs without matching ZCTAs:
['10000', '10041', '10055', '10104', '10105', '10106', '10107', '10118', '10120', '10121', '10122', '10123', '10151', '10155', '10158', '10175', '10176', '10178', '10281', '11251']

SIRTA ZIPs without matching ZCTAs:
['10000', '10281', '11251']


In [38]:
# Flag unverified ZIP codes before integration

housing_df["zip_not_in_zcta"] = (
    housing_df["zip_code"].notna()
    & ~housing_df["zip_code"].isin(valid_zctas)
)

transport_df["zip_not_in_zcta"] = (
    transport_df["zip_code"].notna()
    & ~transport_df["zip_code"].isin(valid_zctas)
)

# Flag ZIP 10000 separately
housing_df["zip_10000_flag"] = (
    housing_df["zip_code"] == "10000"
).fillna(False)

transport_df["zip_10000_flag"] = (
    transport_df["zip_code"] == "10000"
).fillna(False)

print("Housing records with ZIP 10000:",
      housing_df["zip_10000_flag"].sum())

print("SIRTA records with ZIP 10000:",
      transport_df["zip_10000_flag"].sum())

print("\nZIP values absent from the ZCTA reference are flagged.")

Housing records with ZIP 10000: 120
SIRTA records with ZIP 10000: 44

ZIP values absent from the ZCTA reference are flagged.


## 2. Data Aggregation

### Data Aggregation

Before integrating the two datasets, the records are grouped by ZIP code and month.

This is necessary because multiple Housing or SIRTA records may refer to the same geographic area and time period.

For Housing, the aggregated table counts building job application filing records.

For SIRTA, the aggregated table counts transportation safety records. The exact meaning of these counts depends on the dataset's record structure.

Aggregating the datasets separately prevents a many-to-many merge from artificially increasing the counts.


In [39]:
# Remove records without usable integration keys
# Keep valid date/ZIP records, excluding unverified ZIP 10000

housing_valid = housing_df.dropna(
    subset=["zip_code", "date"]
).copy()

transport_valid = transport_df.dropna(
    subset=["zip_code", "date"]
).copy()

housing_excluded_10000 = (
    housing_valid["zip_code"] == "10000"
).sum()

transport_excluded_10000 = (
    transport_valid["zip_code"] == "10000"
).sum()

housing_valid = housing_valid[
    housing_valid["zip_code"] != "10000"
].copy()

transport_valid = transport_valid[
    transport_valid["zip_code"] != "10000"
].copy()

print("Housing records excluded (ZIP 10000):",
      housing_excluded_10000)

print("SIRTA records excluded (ZIP 10000):",
      transport_excluded_10000)

if housing_valid.empty or transport_valid.empty:
    raise ValueError(
        "At least one dataset has no usable ZIP-date records."
    )

# Identify the overlapping monthly period
start_month = max(
    housing_valid["month"].min(),
    transport_valid["month"].min()
)

end_month = min(
    housing_valid["month"].max(),
    transport_valid["month"].max()
)

if start_month > end_month:
    raise ValueError(
        "No overlapping monthly period between the datasets."
    )

print("Integration period:", start_month, "to", end_month)

# Filter both datasets to the common period
housing_valid = housing_valid[
    housing_valid["month"].between(start_month, end_month)
].copy()

transport_valid = transport_valid[
    transport_valid["month"].between(start_month, end_month)
].copy()

print("Housing records:", len(housing_valid))
print("SIRTA records:", len(transport_valid))

Housing records excluded (ZIP 10000): 120
SIRTA records excluded (ZIP 10000): 44
Integration period: 2022-01 to 2026-06
Housing records: 380611
SIRTA records: 12085


### Geographic ZIP Code Validation

ZIP codes were compared with the NYC ZIP Code Tabulation Areas (ZCTA) reference to identify values that could not be geographically verified.

ZIP code `10000` was provisionally excluded from the geographic integration because it was absent from the reference.

Other ZIP codes missing from the ZCTA reference were flagged for investigation rather than automatically deleted, since some legitimate postal ZIP codes do not correspond to ZCTA polygons.

Geographic quality statistics are recalculated using the final cleaned Housing and SIRTA datasets and reported in Section 5.

In [40]:
# Aggregate Housing records by ZIP and month
housing_agg = (
    housing_valid
    .groupby(["zip_code", "month"], as_index=False)
    .agg(
        filing_count=("job_base", "nunique"),
        housing_borough=("borough", "first")
    )
)

# Aggregate SIRTA records by ZIP and month
transport_agg = (
    transport_valid
    .groupby(["zip_code", "month"], as_index=False)
    .agg(
        sirta_count=("open_data_crash_id", "nunique"),
        transport_borough=("borough", "first")
    )
)

print("HOUSING AGGREGATION")
print("Shape:", housing_agg.shape)
display(housing_agg.head())

print("\nSIRTA AGGREGATION")
print("Shape:", transport_agg.shape)
display(transport_agg.head())

HOUSING AGGREGATION
Shape: (10917, 4)


,zip_code,month,filing_count,housing_borough
0,10001,2022-01,212,Manhattan
1,10001,2022-02,171,Manhattan
2,10001,2022-03,225,Manhattan
3,10001,2022-04,186,Manhattan
4,10001,2022-05,217,Manhattan



SIRTA AGGREGATION
Shape: (6022, 4)


,zip_code,month,sirta_count,transport_borough
0,10001,2022-01,1,Manhattan
1,10001,2022-03,1,Manhattan
2,10001,2022-04,1,Manhattan
3,10001,2022-05,1,Manhattan
4,10001,2022-06,2,Manhattan


## 3. Dataset Integration

### Integration Process

After aggregation, the Housing and SIRTA datasets are merged using ZIP code and month.

An outer join is used so that geographic and time combinations from both datasets remain visible.

The merge indicator identifies whether each combination appears in both datasets, only in Housing, or only in SIRTA.

A one-to-one validation check ensures that the aggregated tables contain unique ZIP code-month combinations and that the merge does not multiply the records.


In [41]:
# Merge using ZIP code and month
integrated = pd.merge(
    housing_agg,
    transport_agg,
    on=["zip_code", "month"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

print("INTEGRATION COMPLETED")
print("Shape:", integrated.shape)

print("\nMerge coverage:")
print(integrated["_merge"].value_counts())

display(integrated.head(10))

INTEGRATION COMPLETED
Shape: (10937, 7)

Merge coverage:
_merge
both          6002
left_only     4915
right_only      20
Name: count, dtype: int64


,zip_code,month,filing_count,housing_borough,sirta_count,transport_borough,_merge
0,10001,2022-01,212.0,Manhattan,1.0,Manhattan,both
1,10001,2022-02,171.0,Manhattan,NaN,<NA>,left_only
2,10001,2022-03,225.0,Manhattan,1.0,Manhattan,both
3,10001,2022-04,186.0,Manhattan,1.0,Manhattan,both
4,10001,2022-05,217.0,Manhattan,1.0,Manhattan,both
5,10001,2022-06,186.0,Manhattan,2.0,Manhattan,both
6,10001,2022-07,181.0,Manhattan,NaN,<NA>,left_only
7,10001,2022-08,158.0,Manhattan,1.0,Manhattan,both
8,10001,2022-09,178.0,Manhattan,3.0,Manhattan,both
9,10001,2022-10,157.0,Manhattan,2.0,Manhattan,both


## 4. Post-Integration Cleaning

### Post-Integration Cleaning

After merging, the integrated dataset is checked for missing values, duplicate keys, inconsistent borough labels, and incorrect data types.

The two borough columns are combined into one standardized attribute.

Missing counts caused by unmatched records are preserved as missing values rather than automatically changed to zero.

A source coverage indicator is retained so that the website and research analyses can distinguish between matched and unmatched observations.

The cleaned dataset is then prepared for validation and visualization.


In [42]:
# Examine missing values
print("Missing values before cleaning:")
print(integrated.isna().sum())

# Identify borough disagreements
borough_conflicts = integrated[
    integrated["housing_borough"].notna()
    & integrated["transport_borough"].notna()
    & (
        integrated["housing_borough"]
        != integrated["transport_borough"]
    )
]

print("\nBorough conflicts:", len(borough_conflicts))
display(borough_conflicts.head())

# Combine borough columns
integrated["borough"] = (
    integrated["housing_borough"]
    .combine_first(integrated["transport_borough"])
)

# Flag conflicting assignments for further review
integrated["borough_conflict"] = (
    integrated.index.isin(borough_conflicts.index)
)

# Remove redundant columns
integrated = integrated.drop(
    columns=["housing_borough", "transport_borough"]
)

# Preserve merge information
integrated = integrated.rename(
    columns={"_merge": "data_coverage"}
)

# Standardize data types
integrated["zip_code"] = integrated["zip_code"].astype("string")

integrated["month"] = integrated["month"].astype("string")

integrated["filing_count"] = (
    integrated["filing_count"].astype("Int64")
)

integrated["sirta_count"] = (
    integrated["sirta_count"].astype("Int64")
)

# Inspect remaining issues
print("\nMissing values after cleaning:")
print(integrated.isna().sum())

print("\nDuplicate ZIP-month keys:")
print(
    integrated.duplicated(
        subset=["zip_code", "month"]
    ).sum()
)

display(integrated.head())

Missing values before cleaning:
zip_code                0
month                   0
filing_count           20
housing_borough        20
sirta_count          4915
transport_borough    4915
_merge                  0
dtype: int64

Borough conflicts: 9


,zip_code,month,filing_count,housing_borough,sirta_count,transport_borough,_merge
4790,10463,2022-02,44.0,Manhattan,3.0,Bronx,both
4796,10463,2022-08,54.0,Manhattan,3.0,Bronx,both
4803,10463,2023-03,50.0,Manhattan,2.0,Bronx,both
7902,11237,2023-02,24.0,Brooklyn,1.0,Queens,both
7941,11237,2026-05,28.0,Brooklyn,1.0,Queens,both



Missing values after cleaning:
zip_code               0
month                  0
filing_count          20
sirta_count         4915
data_coverage          0
borough                0
borough_conflict       0
dtype: int64

Duplicate ZIP-month keys:
0


,zip_code,month,filing_count,sirta_count,data_coverage,borough,borough_conflict
0,10001,2022-01,212,1,both,Manhattan,False
1,10001,2022-02,171,<NA>,left_only,Manhattan,False
2,10001,2022-03,225,1,both,Manhattan,False
3,10001,2022-04,186,1,both,Manhattan,False
4,10001,2022-05,217,1,both,Manhattan,False


In [43]:
# Inspect all conflicting borough assignments

print("BOROUGH CONFLICTS")

display(
    borough_conflicts[[
        "zip_code",
        "month",
        "housing_borough",
        "transport_borough"
    ]]
)

BOROUGH CONFLICTS


,zip_code,month,housing_borough,transport_borough
4790,10463,2022-02,Manhattan,Bronx
4796,10463,2022-08,Manhattan,Bronx
4803,10463,2023-03,Manhattan,Bronx
7902,11237,2023-02,Brooklyn,Queens
7941,11237,2026-05,Brooklyn,Queens
8037,11239,2025-05,Brooklyn,Queens
8970,11370,2022-07,Bronx,Queens
9410,11385,2022-12,Queens,Brooklyn
9415,11385,2023-05,Queens,Brooklyn


In [44]:
# Investigate suspicious ZIP code 10000

suspect_zip = "10000"

housing_suspect = housing_df[
    housing_df["zip_code"] == suspect_zip
]

transport_suspect = transport_df[
    transport_df["zip_code"] == suspect_zip
]

print("HOUSING RECORDS WITH ZIP 10000")
print("Count:", len(housing_suspect))

display(
    housing_suspect[
        ["zip", "borough", "filing_date"]
    ].head(10)
)

print("\nSIRTA RECORDS WITH ZIP 10000")
print("Count:", len(transport_suspect))

display(
    transport_suspect[
        ["zip", "borough", "crash_date"]
    ].head(10)
)

HOUSING RECORDS WITH ZIP 10000
Count: 120


,zip,borough,filing_date
20166,10000.0,Manhattan,2022-03-04 00:00:00
22606,10000.0,Manhattan,2022-03-11 00:00:00
24066,10000.0,Manhattan,2022-03-17 00:00:00
24292,10000.0,Manhattan,2022-03-17 00:00:00
27916,10000.0,Manhattan,2022-03-28 00:00:00
29974,10000.0,Manhattan,2022-04-01 00:00:00
34891,10000.0,Manhattan,2022-04-15 00:00:00
36606,10000.0,Manhattan,2022-04-21 00:00:00
40597,10000.0,Manhattan,2022-05-04
44850,10000.0,Manhattan,2022-05-17 00:00:00



SIRTA RECORDS WITH ZIP 10000
Count: 44


,zip,borough,crash_date
3119,10000.0,Manhattan,2022-10-26
3792,10000.0,Manhattan,2023-05-16
4100,10000.0,Manhattan,2023-06-18
4605,10000.0,Manhattan,2023-08-10
5251,10000.0,Manhattan,2024-01-13
7887,10000.0,Manhattan,2025-03-30
7939,10000.0,Manhattan,2025-04-06
7997,10000.0,Manhattan,2025-04-14
8039,10000.0,Manhattan,2025-04-22
8057,10000.0,Manhattan,2025-04-25


## 5. Integration Validation

### Integration Validation

The integrated dataset is validated through several checks:

1. Verify that ZIP code-month combinations are unique.
2. Confirm that Housing filing counts remain unchanged after merging.
3. Confirm that SIRTA record counts remain unchanged after merging.
4. Check for negative counts and invalid values.
5. Examine matched and unmatched records using an integration coverage chart.
6. Compare monthly activity patterns to identify potential anomalies.

These checks help determine whether the integration preserved the information in both datasets correctly.


In [45]:
# Check for duplicate integration keys
assert not integrated.duplicated(
    subset=["zip_code", "month"]
).any(), "Duplicate integration keys detected!"

# Compare original and integrated Housing counts
assert (
    housing_agg["filing_count"].sum()
    == integrated["filing_count"].sum()
), "Housing counts changed during integration!"

# Compare original and integrated SIRTA counts
assert (
    transport_agg["sirta_count"].sum()
    == integrated["sirta_count"].sum()
), "SIRTA counts changed during integration!"

# Check for negative counts
assert (
    integrated["filing_count"].dropna() >= 0
).all(), "Negative Housing counts detected!"

assert (
    integrated["sirta_count"].dropna() >= 0
).all(), "Negative SIRTA counts detected!"

print("All integration checks passed successfully!")

print("Housing filings:", integrated["filing_count"].sum())
print("SIRTA records:", integrated["sirta_count"].sum())

All integration checks passed successfully!
Housing filings: 380611
SIRTA records: 12085


In [46]:
# Visualize integration coverage
coverage = (
    integrated["data_coverage"]
    .value_counts()
    .reset_index()
)

coverage.columns = ["Coverage", "Count"]

fig = px.bar(
    coverage,
    x="Coverage",
    y="Count",
    color="Coverage",
    title="Housing and SIRTA Integration Coverage",
    labels={
        "Coverage": "Source Coverage",
        "Count": "ZIP-Month Combinations"
    }
)

fig.show()

In [47]:
# Calculate integration coverage percentages

coverage_summary = (
    integrated["data_coverage"]
    .value_counts()
    .reset_index()
)

coverage_summary.columns = [
    "Merge Category",
    "ZIP-Month Combinations"
]

coverage_summary["Percentage"] = (
    coverage_summary["ZIP-Month Combinations"]
    / len(integrated) * 100
).round(2)

display(coverage_summary)

,Merge Category,ZIP-Month Combinations,Percentage
0,both,6002,54.88
1,left_only,4915,44.94
2,right_only,20,0.18


### Integration Coverage Results

The final integration uses Member 1's cleaned Housing dataset and Member 2's final SIRTA dataset, including ZIP codes recovered through spatial matching.

Both datasets are standardized by ZIP code and month before aggregation. An outer join preserves combinations represented in either dataset.

The integration validation checks for duplicate ZIP-month combinations, preserved record counts, missing geographic identifiers, and conflicting borough assignments.

Records without usable ZIP codes or dates are excluded from the geographic integration, but are retained in the original cleaned datasets for coverage analysis.

ZIP code `10000` is provisionally excluded because it could not be verified in the NYC ZCTA reference.

The final integration counts and coverage percentages are reported in the validation tables generated above.

In [48]:
# Calculate missing ZIP statistics

zip_quality = pd.DataFrame({
    "Dataset": ["Housing", "SIRTA"],
    "Total Records": [
        len(housing_df),
        len(transport_df)
    ],
    "Missing ZIP": [
        housing_df["zip_code"].isna().sum(),
        transport_df["zip_code"].isna().sum()
    ]
})

zip_quality["Missing ZIP %"] = (
    zip_quality["Missing ZIP"]
    / zip_quality["Total Records"] * 100
).round(2)

display(zip_quality)

,Dataset,Total Records,Missing ZIP,Missing ZIP %
0,Housing,381142,411,0.11
1,SIRTA,13309,1180,8.87


In [49]:
# Geographic validation summary

geo_summary = pd.DataFrame({
    "Check": [
        "Housing records with ZIP 10000",
        "SIRTA records with ZIP 10000",
        "Housing ZIPs outside ZCTA reference",
        "SIRTA ZIPs outside ZCTA reference",
        "Borough conflicts after integration"
    ],
    "Count": [
        housing_df["zip_10000_flag"].sum(),
        transport_df["zip_10000_flag"].sum(),
        housing_df["zip_not_in_zcta"].sum(),
        transport_df["zip_not_in_zcta"].sum(),
        integrated["borough_conflict"].sum()
    ]
})

display(geo_summary)

,Check,Count
0,Housing records with ZIP 10000,120
1,SIRTA records with ZIP 10000,44
2,Housing ZIPs outside ZCTA reference,2576
3,SIRTA ZIPs outside ZCTA reference,48
4,Borough conflicts after integration,9


In [50]:
# Crash ZIP coverage report by year
# Includes fatal crashes missing geographic ZIP codes

crash_coverage = transport_df.copy()

crash_coverage["year"] = crash_coverage["date"].dt.year
crash_coverage["fatal_crash"] = (
    pd.to_numeric(
        crash_coverage["total_fatalities"],
        errors="coerce"
    ).fillna(0) > 0
)

crash_coverage["missing_zip"] = (
    crash_coverage["zip_code"].isna()
)

crash_coverage["fatal_missing_zip"] = (
    crash_coverage["fatal_crash"]
    & crash_coverage["missing_zip"]
)

coverage_report = (
    crash_coverage
    .dropna(subset=["year"])
    .groupby("year", as_index=False)
    .agg(
        total_crashes=("open_data_crash_id", "nunique"),
        crashes_missing_zip=("missing_zip", "sum"),
        fatal_crashes=("fatal_crash", "sum"),
        fatal_crashes_missing_zip=("fatal_missing_zip", "sum")
    )
)

coverage_report["year"] = (
    coverage_report["year"].astype(int)
)

coverage_report["crash_missing_zip_pct"] = (
    100
    * coverage_report["crashes_missing_zip"]
    / coverage_report["total_crashes"]
).round(2)

coverage_report["fatal_crash_missing_zip_pct"] = (
    100
    * coverage_report["fatal_crashes_missing_zip"]
    / coverage_report["fatal_crashes"].replace(0, float("nan"))
).round(2)

display(coverage_report)

coverage_report.to_csv(
    "crash_zip_coverage_by_year.csv",
    index=False
)

print("Crash coverage report exported.")

,year,total_crashes,crashes_missing_zip,fatal_crashes,fatal_crashes_missing_zip,crash_missing_zip_pct,fatal_crash_missing_zip_pct
0,2022,2879,303,245,39,10.52,15.92
1,2023,3016,311,237,105,10.31,44.30
2,2024,3116,381,232,93,12.23,40.09
3,2025,2947,116,211,64,3.94,30.33
4,2026,1351,69,121,53,5.11,43.80


Crash coverage report exported.


## 6. Research Question 5


**Do housing filings and SIRTA transportation safety records show similar monthly patterns across NYC boroughs?**

This question examines whether changes in housing construction activity are associated with changes in transportation safety records over time.

The analysis uses an interactive line chart to compare monthly activity across boroughs.

A lag-correlation analysis is also conducted to explore whether changes in Housing activity tend to occur before, after, or during similar changes in SIRTA records.

The results are exploratory and do not establish causation.


In [51]:
# Use matched records with consistent geographic data
matched = integrated[
    (integrated["data_coverage"] == "both")
    & (~integrated["borough_conflict"])
    & (integrated["borough"].notna())
].copy()

# Aggregate to borough-month level
monthly_borough = (
    matched
    .groupby(["borough", "month"], as_index=False)
    .agg(
        filing_count=("filing_count", "sum"),
        sirta_count=("sirta_count", "sum")
    )
)

monthly_borough["month"] = pd.to_datetime(
    monthly_borough["month"]
)

print("Borough-month observations:", len(monthly_borough))

display(monthly_borough.head())

Borough-month observations: 270


,borough,month,filing_count,sirta_count
0,Bronx,2022-01-01,690,36
1,Bronx,2022-02-01,600,33
2,Bronx,2022-03-01,796,37
3,Bronx,2022-04-01,621,29
4,Bronx,2022-05-01,902,41


In [52]:
# Reshape for Plotly
chart_data = monthly_borough.melt(
    id_vars=["borough", "month"],
    value_vars=["filing_count", "sirta_count"],
    var_name="dataset",
    value_name="count"
)

# Create interactive monthly chart
fig = px.line(
    chart_data,
    x="month",
    y="count",
    color="dataset",
    facet_row="borough",
    title="Monthly Housing Filings vs SIRTA Records",
    markers=True,
    height=max(
        500,
        monthly_borough["borough"].nunique() * 250
    )
)

fig.update_yaxes(matches=None)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    legend_title="Dataset"
)

fig.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig.show()

In [53]:
lag_results = []

for borough, group in monthly_borough.groupby("borough"):

    group = group.sort_values("month").set_index("month")

    # Preserve missing months as missing values
    group = group.asfreq("MS")

    for lag in range(-3, 4):

        # Positive lag: filings precede SIRTA records
        shifted = group["filing_count"].shift(lag)

        paired = pd.DataFrame({
            "filings": shifted,
            "sirta": group["sirta_count"]
        }).dropna()

        if (
            len(paired) >= 6
            and paired["filings"].nunique() > 1
            and paired["sirta"].nunique() > 1
        ):
            corr = paired["filings"].corr(paired["sirta"])
        else:
            corr = np.nan

        lag_results.append({
            "borough": borough,
            "lag": lag,
            "correlation": corr,
            "observations": len(paired)
        })

lag_df = pd.DataFrame(lag_results)

fig = px.line(
    lag_df,
    x="lag",
    y="correlation",
    color="borough",
    markers=True,
    hover_data=["observations"],
    title="Housing–SIRTA Lag Correlation by Borough",
    labels={
        "lag": "Lag (Months)",
        "correlation": "Pearson Correlation"
    }
)

fig.add_hline(y=0, line_dash="dash")

fig.show()

display(lag_df)

,borough,lag,correlation,observations
0,Bronx,-3,-0.134149,51
1,Bronx,-2,-0.105340,52
2,Bronx,-1,-0.005720,53
3,Bronx,0,0.414932,54
4,Bronx,1,0.225710,53
5,Bronx,2,0.320715,52
6,Bronx,3,0.110744,51
7,Brooklyn,-3,-0.071314,51
8,Brooklyn,-2,-0.119942,52
9,Brooklyn,-1,0.053648,53


## 7. Research Question 6


**Which ZIP codes have more or fewer SIRTA transportation safety records than expected based on their housing filing activity?**

This question investigates differences in the relationship between housing development activity and transportation safety records across NYC ZIP codes.

A scatter plot is used to compare the two measures.

A simple linear regression model estimates the general relationship between the variables.

Residuals are then calculated to identify ZIP codes with observed SIRTA counts substantially above or below the fitted values.

Large positive residuals indicate higher observed counts than the model estimates, while large negative residuals indicate lower observed counts.

These findings are exploratory and should not be interpreted as direct measures of neighborhood safety. The model does not control for population, traffic exposure, or other important factors.


### Analytical Sample Limitations

Research Questions 5 and 6 currently use ZIP-month combinations that appear in both datasets.

Although this provides comparable observations for both Housing and SIRTA, it excludes locations or months represented in only one dataset. This may affect the observed relationships and regression estimates.

The analysis also uses counts rather than population-adjusted or traffic-adjusted rates.

Consequently, the results should be interpreted as exploratory associations rather than evidence that housing activity directly causes changes in transportation safety.

The final cleaned SIRTA dataset is used here, but this matched-only analysis remains exploratory.

In [54]:
# RQ6: Compare Housing and SIRTA activity by ZIP code
# Use complete years 2022-2025, as required by DATA_CONTRACT.md

# Use standardized source records, not matched ZIP-month rows
h_rq6 = housing_df[
    housing_df["date"].dt.year.between(2022, 2025)
    & housing_df["zip_code"].notna()
    & housing_df["zip_code"].ne("10000").fillna(False)
].copy()

t_rq6 = transport_df[
    transport_df["date"].dt.year.between(2022, 2025)
    & transport_df["zip_code"].notna()
    & transport_df["zip_code"].ne("10000").fillna(False)
].copy()

# Count unique Housing jobs per ZIP
h_counts = (
    h_rq6.groupby("zip_code")["job_base"]
    .nunique()
    .rename("filing_count")
)

# Count unique crashes per ZIP
t_counts = (
    t_rq6.groupby("zip_code")["open_data_crash_id"]
    .nunique()
    .rename("sirta_count")
)

# Combine both datasets, keeping all usable ZIP codes
zip_analysis = pd.concat(
    [h_counts, t_counts],
    axis=1
).fillna(0).reset_index()

# Assign a borough using the most common nonmissing value
def most_common_borough(series):
    values = series.dropna().mode()
    return values.iloc[0] if not values.empty else pd.NA

h_boroughs = (
    h_rq6.groupby("zip_code")["borough"]
    .agg(most_common_borough)
)

t_boroughs = (
    t_rq6.groupby("zip_code")["borough"]
    .agg(most_common_borough)
)

zip_analysis["borough"] = (
    zip_analysis["zip_code"].map(h_boroughs)
    .combine_first(zip_analysis["zip_code"].map(t_boroughs))
)

# 2022-2025 = 4 complete years = 48 calendar months
zip_analysis["months_observed"] = 48

# Average activity per calendar month
zip_analysis["filings_per_month"] = (
    zip_analysis["filing_count"] / 48
)

zip_analysis["sirta_per_month"] = (
    zip_analysis["sirta_count"] / 48
)

# Verify the regression has enough usable observations
print("ZIP codes available for RQ6:", len(zip_analysis))

print(
    "Unique Housing activity values:",
    zip_analysis["filings_per_month"].nunique()
)

assert len(zip_analysis) >= 3, (
    "Not enough ZIP codes for regression."
)

assert zip_analysis["filings_per_month"].nunique() >= 2, (
    "Not enough variation in Housing activity."
)

display(zip_analysis.head(10))

ZIP codes available for RQ6: 226
Unique Housing activity values: 209


,zip_code,filing_count,sirta_count,borough,months_observed,filings_per_month,sirta_per_month
0,10001,6664.0,83.0,Manhattan,48,138.833333,1.729167
1,10002,3191.0,104.0,Manhattan,48,66.479167,2.166667
2,10003,6266.0,61.0,Manhattan,48,130.541667,1.270833
3,10004,1092.0,10.0,Manhattan,48,22.750000,0.208333
4,10005,1371.0,5.0,Manhattan,48,28.562500,0.104167
5,10006,715.0,4.0,Manhattan,48,14.895833,0.083333
6,10007,1793.0,14.0,Manhattan,48,37.354167,0.291667
7,10009,2298.0,57.0,Manhattan,48,47.875000,1.187500
8,10010,4103.0,55.0,Manhattan,48,85.479167,1.145833
9,10011,6567.0,54.0,Manhattan,48,136.812500,1.125000


In [55]:
# Diagnose Research Question 6

print("Total integrated rows:", len(integrated))
print("Matched rows:", len(matched))
print("ZIP codes after RQ6 filtering:", len(zip_analysis))

print("\nUnique Housing activity values:",
      zip_analysis["filings_per_month"].nunique())

print("\nMonths observed per ZIP:")
print(zip_analysis["months_observed"].describe())

print("\nMissing regression values:")
print(
    zip_analysis[
        ["filings_per_month", "sirta_per_month"]
    ].isna().sum()
)

display(zip_analysis.head(10))

Total integrated rows: 10937
Matched rows: 5993
ZIP codes after RQ6 filtering: 226

Unique Housing activity values: 209

Months observed per ZIP:
count    226.0
mean      48.0
std        0.0
min       48.0
25%       48.0
50%       48.0
75%       48.0
max       48.0
Name: months_observed, dtype: float64

Missing regression values:
filings_per_month    0
sirta_per_month      0
dtype: int64


,zip_code,filing_count,sirta_count,borough,months_observed,filings_per_month,sirta_per_month
0,10001,6664.0,83.0,Manhattan,48,138.833333,1.729167
1,10002,3191.0,104.0,Manhattan,48,66.479167,2.166667
2,10003,6266.0,61.0,Manhattan,48,130.541667,1.270833
3,10004,1092.0,10.0,Manhattan,48,22.750000,0.208333
4,10005,1371.0,5.0,Manhattan,48,28.562500,0.104167
5,10006,715.0,4.0,Manhattan,48,14.895833,0.083333
6,10007,1793.0,14.0,Manhattan,48,37.354167,0.291667
7,10009,2298.0,57.0,Manhattan,48,47.875000,1.187500
8,10010,4103.0,55.0,Manhattan,48,85.479167,1.145833
9,10011,6567.0,54.0,Manhattan,48,136.812500,1.125000


In [56]:
# Prepare regression variables
x = zip_analysis["filings_per_month"].to_numpy(
    dtype=float
)

y = zip_analysis["sirta_per_month"].to_numpy(
    dtype=float
)

if len(x) < 3 or np.unique(x).size < 2:
    raise ValueError(
        "Not enough observations for a meaningful regression."
    )

# Fit linear regression
slope, intercept = np.polyfit(x, y, 1)

# Predicted SIRTA records per month
zip_analysis["predicted_sirta"] = (
    slope * zip_analysis["filings_per_month"] + intercept
)

# Calculate residuals
zip_analysis["residual"] = (
    zip_analysis["sirta_per_month"]
    - zip_analysis["predicted_sirta"]
)

# Classify the most extreme observed residuals
zip_analysis["category"] = "Other"

top_high = zip_analysis.nlargest(
    5, "residual"
).index

top_low = zip_analysis.nsmallest(
    5, "residual"
).index

zip_analysis.loc[top_high, "category"] = (
    "Higher than estimated"
)

zip_analysis.loc[top_low, "category"] = (
    "Lower than estimated"
)

print("Regression slope:", round(slope, 4))
print("Regression intercept:", round(intercept, 4))

display(
    zip_analysis.sort_values(
        "residual", ascending=False
    ).head(10)
)

Regression slope: 0.0146
Regression intercept: 0.5293


,zip_code,filing_count,sirta_count,borough,months_observed,filings_per_month,sirta_per_month,predicted_sirta,residual,category
163,11236,1721.0,175.0,Brooklyn,48,35.854167,3.645833,1.054282,2.591551,Higher than estimated
161,11234,1957.0,153.0,Brooklyn,48,40.770833,3.187500,1.126279,2.061221,Higher than estimated
135,11207,2928.0,156.0,Brooklyn,48,61.000000,3.250000,1.422500,1.827500,Higher than estimated
97,10456,1826.0,137.0,Bronx,48,38.041667,2.854167,1.086315,1.767852,Higher than estimated
140,11212,2482.0,146.0,Brooklyn,48,51.708333,3.041667,1.286440,1.755227,Higher than estimated
108,10467,2065.0,135.0,Bronx,48,43.020833,2.812500,1.159226,1.653274,Other
98,10457,1903.0,132.0,Bronx,48,39.645833,2.750000,1.109805,1.640195,Other
95,10454,1102.0,120.0,Bronx,48,22.958333,2.500000,0.865445,1.634555,Other
131,11203,1694.0,128.0,Brooklyn,48,35.291667,2.666667,1.046045,1.620621,Other
94,10453,1579.0,125.0,Bronx,48,32.895833,2.604167,1.010963,1.593204,Other


In [57]:
# Create scatter plot
fig = px.scatter(
    zip_analysis,
    x="filings_per_month",
    y="sirta_per_month",
    color="category",
    hover_name="zip_code",
    hover_data=[
        "borough",
        "months_observed",
        "predicted_sirta",
        "residual"
    ],
    title="Housing Activity vs SIRTA Activity by ZIP Code",
    labels={
        "filings_per_month": "Average Housing Filings per Month",
        "sirta_per_month": "Average SIRTA Records per Month"
    }
)

# Add regression line
x_line = np.linspace(x.min(), x.max(), 100)

fig.add_trace(
    go.Scatter(
        x=x_line,
        y=slope * x_line + intercept,
        mode="lines",
        name="Regression Line",
        line=dict(dash="dash", color="black")
    )
)

fig.update_layout(height=600)

fig.show()

In [58]:
print("TOP 5 ZIP CODES ABOVE THE REGRESSION LINE")

display(
    zip_analysis.nlargest(
        5, "residual"
    )[[
        "zip_code",
        "borough",
        "filings_per_month",
        "sirta_per_month",
        "predicted_sirta",
        "residual"
    ]]
)

print("\nTOP 5 ZIP CODES BELOW THE REGRESSION LINE")

display(
    zip_analysis.nsmallest(
        5, "residual"
    )[[
        "zip_code",
        "borough",
        "filings_per_month",
        "sirta_per_month",
        "predicted_sirta",
        "residual"
    ]]
)

TOP 5 ZIP CODES ABOVE THE REGRESSION LINE


,zip_code,borough,filings_per_month,sirta_per_month,predicted_sirta,residual
163,11236,Brooklyn,35.854167,3.645833,1.054282,2.591551
161,11234,Brooklyn,40.770833,3.187500,1.126279,2.061221
135,11207,Brooklyn,61.000000,3.250000,1.422500,1.827500
97,10456,Bronx,38.041667,2.854167,1.086315,1.767852
140,11212,Brooklyn,51.708333,3.041667,1.286440,1.755227



TOP 5 ZIP CODES BELOW THE REGRESSION LINE


,zip_code,borough,filings_per_month,sirta_per_month,predicted_sirta,residual
19,10022,Manhattan,176.000000,1.604167,3.106479,-1.502313
12,10014,Manhattan,107.354167,0.625000,2.101278,-1.476278
9,10011,Manhattan,136.812500,1.125000,2.532645,-1.407645
2,10003,Manhattan,130.541667,1.270833,2.440819,-1.169986
11,10013,Manhattan,118.020833,1.125000,2.257473,-1.132473


## 8. Export Final Integrated Dataset

### Final Integrated Dataset

After completing the integration and validation stages, the integrated dataset is exported as a CSV file.

The final file contains ZIP codes, boroughs, months, Housing filing counts, SIRTA record counts, and source coverage information.

This dataset will be provided to the website development team for interactive filtering, visualization, and reporting.

The integrated file is also retained as an output of the data engineering pipeline to support reproducibility.


In [59]:
# Select final columns
final_integrated = integrated[[
    "zip_code",
    "borough",
    "month",
    "filing_count",
    "sirta_count",
    "data_coverage",
    "borough_conflict"
]].copy()

# Export dataset
filename = "NYC_Integrated_Dataset_preliminary.csv"

final_integrated.to_csv(
    filename,
    index=False
)

print("Dataset exported successfully!")
print("Final shape:", final_integrated.shape)

display(final_integrated.head())

# Download from Colab
# files.download(filename)  # optional: preliminary exploratory export

Dataset exported successfully!
Final shape: (10937, 7)


,zip_code,borough,month,filing_count,sirta_count,data_coverage,borough_conflict
0,10001,Manhattan,2022-01,212,1,both,False
1,10001,Manhattan,2022-02,171,<NA>,left_only,False
2,10001,Manhattan,2022-03,225,1,both,False
3,10001,Manhattan,2022-04,186,1,both,False
4,10001,Manhattan,2022-05,217,1,both,False


In [60]:
print("========== MEMBER 3 FINAL CHECK ==========")

print("\n1. DATASET SIZES")
print("Housing records:", len(housing))
print("Crash records:", len(transport))
print("Integrated records:", len(integrated))

print("\n2. INTEGRATION PERIOD")
print("Start:", start_month)
print("End:", end_month)

print("\n3. MERGE COVERAGE")
print(integrated["data_coverage"].value_counts())

print("\n4. MISSING VALUES")
print(integrated.isnull().sum())

print("\n5. DUPLICATE INTEGRATION KEYS")
print(
    integrated.duplicated(
        subset=["zip_code", "month"]
    ).sum()
)

print("\n6. BOROUGH CONFLICTS")
print(integrated["borough_conflict"].sum())

print("\n7. RESEARCH QUESTION DATA")
print("RQ5 borough-month observations:", len(monthly_borough))
print("RQ6 ZIP codes:", len(zip_analysis))

print("\n8. FINAL DATASET")
print("Shape:", final_integrated.shape)
display(final_integrated.head(10))

print("\n========== CHECK COMPLETE ==========")

========== MEMBER 3 FINAL CHECK ==========

1. DATASET SIZES
Housing records: 381142
Crash records: 13309
Integrated records: 10937

2. INTEGRATION PERIOD
Start: 2022-01
End: 2026-06

3. MERGE COVERAGE
data_coverage
both          6002
left_only     4915
right_only      20
Name: count, dtype: int64

4. MISSING VALUES
zip_code               0
month                  0
filing_count          20
sirta_count         4915
data_coverage          0
borough                0
borough_conflict       0
dtype: int64

5. DUPLICATE INTEGRATION KEYS
0

6. BOROUGH CONFLICTS
9

7. RESEARCH QUESTION DATA
RQ5 borough-month observations: 270
RQ6 ZIP codes: 226

8. FINAL DATASET
Shape: (10937, 7)


,zip_code,borough,month,filing_count,sirta_count,data_coverage,borough_conflict
0,10001,Manhattan,2022-01,212,1,both,False
1,10001,Manhattan,2022-02,171,<NA>,left_only,False
2,10001,Manhattan,2022-03,225,1,both,False
3,10001,Manhattan,2022-04,186,1,both,False
4,10001,Manhattan,2022-05,217,1,both,False
5,10001,Manhattan,2022-06,186,2,both,False
6,10001,Manhattan,2022-07,181,<NA>,left_only,False
7,10001,Manhattan,2022-08,158,1,both,False
8,10001,Manhattan,2022-09,178,3,both,False
9,10001,Manhattan,2022-10,157,2,both,False



========== CHECK COMPLETE ==========


## 9. Website-Compatible Exports (Data Contract v2)

**The earlier ZIP-month dataset is retained for exploratory RQ5/RQ6 analysis only.** The contract requires two separate grids: ZIP by complete year (2022–2025) and borough by month (2022-01 through 2026-06).

The ZIP-to-borough lookup uses the most frequent Housing borough for that ZIP (fallback: SIRTA), because a polygon-to-borough lookup was not supplied. The lookup is exported for auditing. ZIP 10000 is provisionally excluded from ZIP-level aggregation because it was not verified against the NYC ZCTA reference. Other ZIP values are retained provisionally. Counts with no source records are set to zero, but the permit-to-signoff median remains missing. Missing-ZIP crashes are retained in the borough-month table when a borough is known.

**Pending team decisions:** H5 `No Work` inclusion in total deduped jobs and the precise ZIP-to-borough mapping should be confirmed by Member 1 / the contract owner. Spatial ZIP agreement of 91.9% is below the 95% target; describe this limitation alongside geographic findings.


In [61]:
"""Member 3 contract v2 exports. Requires pandas and numpy. Colab: %run ... or paste into cell."""
from pathlib import Path
import numpy as np
import pandas as pd

BOROUGHS = ['Bronx','Brooklyn','Manhattan','Queens','Staten Island']
H_METRICS = ['h_filings_total','h_new_building','h_demolition','h_alteration_co','h_proposed_units','h_sidewalk_shed','h_curb_cut']
H_MEDIAN = 'h_median_days_permit_to_signoff'
T_METRICS = ['t_crashes','t_crashes_street','t_severe_injuries','t_non_severe_injuries','t_fatalities','t_fatal_crashes','t_repeat_location_crashes','t_highway_crashes']

def clean_zip(s):
    return s.astype('string').str.strip().str.extract(r'^(\d{5})(?:\.0)?$',expand=False).astype('string')

def as_bool(s):
    return s.astype('string').str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False}).fillna(False).astype(bool)

def count_fatal(s):
    return pd.to_numeric(s,errors='coerce').fillna(0).gt(0)

def make_exports(housing,crash,output_dir='data/clean'):
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    h=housing.copy();t=crash.copy()
    h.columns=h.columns.str.strip().str.lower()
    t.columns=t.columns.str.strip().str.lower()
    h_required=['job_base','job_type','filing_date','borough','zip','proposed_dwelling_units','days_permit_to_signoff','is_sidewalk_shed','is_curb_cut']
    t_required=['open_data_crash_id','crash_date','borough','zip','zip_source','total_severe_injuries','total_non_severe_injuries','total_fatalities','is_highway','is_repeat_location']
    for label,df,required in [('Housing',h,h_required),('SIRTA',t,t_required)]:
        missing=set(required)-set(df.columns)
        if missing: raise ValueError(f'{label}: missing required columns: {sorted(missing)}')
    assert h['job_base'].notna().all() and h['job_base'].is_unique, 'Housing job_base must be non-null unique'
    assert t['open_data_crash_id'].notna().all() and t['open_data_crash_id'].is_unique, 'Crash ID must be non-null unique'
    for df,col in [(h,'filing_date'),(t,'crash_date')]:
        df['date']=pd.to_datetime(df[col],format='mixed',errors='coerce')
        df['year']=df['date'].dt.year.astype('Int64')
        df['month']=df['date'].dt.month.astype('Int64')
        df['year_month']=df['date'].dt.strftime('%Y-%m')
        df['zip']=clean_zip(df['zip'])
        df['borough']=df['borough'].astype('string').str.strip()
        df.loc[~df['borough'].isin(BOROUGHS),'borough']=pd.NA
    # ZIP 10000 is unverified against the NYC ZCTA boundary reference.
    h['geo_zip']=h['zip'].mask(h['zip'].eq('10000').fillna(False))
    t['geo_zip']=t['zip'].mask(t['zip'].eq('10000').fillna(False))
    h['h_filings_total']=1
    h['h_new_building']=h['job_type'].eq('New Building').fillna(False).astype(int)
    h['h_demolition']=h['job_type'].eq('Full Demolition').fillna(False).astype(int)
    h['h_alteration_co']=h['job_type'].eq('Alteration CO').fillna(False).astype(int)
    h['h_proposed_units']=pd.to_numeric(h['proposed_dwelling_units'],errors='coerce').where(h['job_type'].eq('New Building'),0).fillna(0)
    h['h_sidewalk_shed']=as_bool(h['is_sidewalk_shed']).astype(int)
    h['h_curb_cut']=as_bool(h['is_curb_cut']).astype(int)
    h[H_MEDIAN]=pd.to_numeric(h['days_permit_to_signoff'],errors='coerce').where(h['job_type'].eq('New Building'))
    t['t_crashes']=1
    highway=as_bool(t['is_highway'])
    t['t_crashes_street']=(~highway).astype(int)
    for source,target in [('total_severe_injuries','t_severe_injuries'),('total_non_severe_injuries','t_non_severe_injuries'),('total_fatalities','t_fatalities')]:
        t[target]=pd.to_numeric(t[source],errors='coerce').fillna(0)
    t['t_fatal_crashes']=count_fatal(t['total_fatalities']).astype(int)
    t['t_repeat_location_crashes']=as_bool(t['is_repeat_location']).astype(int)
    t['t_highway_crashes']=highway.astype(int)
    # Complete year window is deliberately restricted to complete years (2022–2025).
    hy=h[h['year'].between(2022,2025) & h['geo_zip'].notna()].copy()
    ty=t[t['year'].between(2022,2025) & t['geo_zip'].notna()].copy()
    # ZIP→borough lookup: modal Housing borough per ZIP; fallback to modal crash borough.
    # This is a source-derived lookup rather than an authoritative polygon lookup.
    def modes(df):
        d=df.dropna(subset=['geo_zip','borough'])
        if d.empty:return pd.Series(dtype='string')
        ct=d.groupby(['geo_zip','borough']).size().rename('n').reset_index()
        return ct.sort_values(['geo_zip','n','borough'],ascending=[True,False,True]).drop_duplicates('geo_zip').set_index('geo_zip')['borough']
    hmap=modes(h);tmap=modes(t)
    all_zip=sorted(set(hy['geo_zip'])|set(ty['geo_zip']))
    lookup=pd.DataFrame({'zip':all_zip})
    lookup['borough']=lookup['zip'].map(hmap).fillna(lookup['zip'].map(tmap)).astype('string')
    if lookup['borough'].isna().any():
        raise ValueError('ZIPs lacking borough: '+str(lookup.loc[lookup['borough'].isna(),'zip'].tolist()))
    lookup.to_csv(out/'zip_borough_lookup_audit.csv',index=False)
    grid=pd.MultiIndex.from_product([all_zip,range(2022,2026)],names=['zip','year']).to_frame(index=False)
    grid=grid.merge(lookup,on='zip',how='left',validate='many_to_one')
    def agg(df,keys,metrics,median=False):
        if df.empty:return pd.DataFrame(columns=keys+metrics+([H_MEDIAN] if median else []))
        named={m:(m,'sum') for m in metrics}
        if median:named[H_MEDIAN]=(H_MEDIAN,'median')
        return df.groupby(keys,as_index=False,dropna=False).agg(**named)
    ha=agg(hy,['geo_zip','year'],H_METRICS,median=True).rename(columns={'geo_zip':'zip'})
    ta=agg(ty,['geo_zip','year'],T_METRICS).rename(columns={'geo_zip':'zip'})
    zy=grid.merge(ha,on=['zip','year'],how='left',validate='one_to_one').merge(ta,on=['zip','year'],how='left',validate='one_to_one')
    for c in H_METRICS+T_METRICS:zy[c]=zy[c].fillna(0)
    zy=zy[['zip','borough','year']+H_METRICS+[H_MEDIAN]+T_METRICS]
    zy['year']=zy['year'].astype(int)
    assert zy['zip'].str.fullmatch(r'\d{5}').all() and zy['zip'].notna().all()
    assert zy['borough'].isin(BOROUGHS).all() and zy[['zip','year']].duplicated().sum()==0
    assert len(zy)==4*len(all_zip)
    for m in H_METRICS+T_METRICS:assert zy[m].ge(0).all(),m
    for year in range(2022,2026):
        assert zy.loc[zy.year.eq(year),'h_filings_total'].sum()==hy.loc[hy.year.eq(year)].shape[0]
        assert zy.loc[zy.year.eq(year),'t_crashes'].sum()==ty.loc[ty.year.eq(year)].shape[0]
    zy.to_csv(out/'zip_year_integrated.csv',index=False)
    # Monthly series includes ZIP-missing crashes with known borough.
    months=pd.period_range('2022-01','2026-06',freq='M')
    bm_grid=pd.MultiIndex.from_product([BOROUGHS,months],names=['borough','period']).to_frame(index=False)
    bm_grid['year']=bm_grid['period'].dt.year.astype(int)
    bm_grid['month']=bm_grid['period'].dt.month.astype(int)
    bm_grid['year_month']=bm_grid['period'].astype(str)
    bm_grid=bm_grid.drop(columns='period')
    hm=h[h['year_month'].between('2022-01','2026-06') & h['borough'].notna()].copy()
    tm=t[t['year_month'].between('2022-01','2026-06') & t['borough'].notna()].copy()
    keys=['borough','year','month','year_month']
    bma=agg(hm,keys,H_METRICS)
    bmt=agg(tm,keys,T_METRICS)
    bm=bm_grid.merge(bma,on=keys,how='left',validate='one_to_one').merge(bmt,on=keys,how='left',validate='one_to_one')
    for c in H_METRICS+T_METRICS:bm[c]=bm[c].fillna(0)
    bm=bm[keys+H_METRICS+T_METRICS]
    assert len(bm)==270 and not bm[keys[:1]+['year_month']].duplicated().any()
    assert bm['h_filings_total'].sum()==len(hm)
    assert bm['t_crashes'].sum()==len(tm)
    bm.to_csv(out/'borough_month_integrated.csv',index=False)
    # Coverage report per specified dimensions; explicit all-source totals and excluded counts.
    t['zip_source']=t['zip_source'].astype('string').fillna('missing')
    # Report the source's original missingness separately from provisional exclusion of ZIP 10000.
    t['zip_source_report']=t['zip_source']
    t['fatal']=np.where(count_fatal(t['total_fatalities']),'fatal','nonfatal')
    t['highway']=np.where(highway,'highway','street')
    cov=[]
    def add_coverage(group,dimension,level):
        sources=group['zip_source_report'].value_counts()
        n=len(group)
        row={'dimension':dimension,'group':str(level),'total_crashes':n}
        for s in ['original','spatial_join','missing']:
            v=int(sources.get(s,0));row[f'{s}_count']=v;row[f'{s}_pct']=round(100*v/n,2) if n else np.nan
        row['fatal_crashes']=int(group['t_fatal_crashes'].sum())
        row['fatal_crashes_missing_zip']=int(group.loc[group['zip'].isna(),'t_fatal_crashes'].sum())
        row['fatal_crashes_missing_zip_pct']=round(100*row['fatal_crashes_missing_zip']/row['fatal_crashes'],2) if row['fatal_crashes'] else np.nan
        row['housing_rows_dropped_zip_year']=np.nan
        row['crash_rows_dropped_zip_year']=np.nan
        row['unverified_zip_10000_crashes']=np.nan
        cov.append(row)
    add_coverage(t,'overall','all')
    for dim,column in [('year','year'),('borough','borough'),('is_highway','highway'),('fatal_status','fatal')]:
        for key,part in t.groupby(column,dropna=False):add_coverage(part,dim,key if pd.notna(key) else 'missing')
    report=pd.DataFrame(cov)
    # Distinguish year-window losses from missing location losses.
    report.loc[report['dimension'].eq('overall'),'housing_rows_dropped_zip_year']=int((h['year'].between(2022,2025) & h['geo_zip'].isna()).sum())
    report.loc[report['dimension'].eq('overall'),'crash_rows_dropped_zip_year']=int((t['year'].between(2022,2025) & t['geo_zip'].isna()).sum())
    report.loc[report['dimension'].eq('overall'),'unverified_zip_10000_crashes']=int(t['zip'].eq('10000').sum())
    report.to_csv(out/'coverage_report.csv',index=False)
    print('ZIP-year:',zy.shape,'borough-month:',bm.shape,'coverage:',report.shape)
    print('Annual retained housing:',len(hy),'crashes:',len(ty),' ZIPs:',len(all_zip))
    print('Borough-month crash rows:',len(tm),'missing borough excluded:',len(t)-len(tm))
    print('Overall fatal crash missing ZIP percentage:',report.loc[0,'fatal_crashes_missing_zip_pct'])
    print('Outputs:',out.resolve())
    return zy,bm,report,lookup


# Build the contract CSVs from the final cleaned datasets
zip_year_integrated, borough_month_integrated, coverage_report_v2, zip_borough_lookup = make_exports(housing, transport, '/content/data/clean')
display(zip_year_integrated.head())
display(borough_month_integrated.head())
display(coverage_report_v2.head())


/tmp/ipykernel_9185/2938796865.py:15: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



ZIP-year: (904, 19) borough-month: (270, 19) coverage: (16, 15)
Annual retained housing: 346206 crashes: 10811  ZIPs: 226
Borough-month crash rows: 13299 missing borough excluded: 10
Overall fatal crash missing ZIP percentage: 33.84
Outputs: /content/data/clean


,zip,borough,year,h_filings_total,h_new_building,h_demolition,h_alteration_co,h_proposed_units,h_sidewalk_shed,h_curb_cut,h_median_days_permit_to_signoff,t_crashes,t_crashes_street,t_severe_injuries,t_non_severe_injuries,t_fatalities,t_fatal_crashes,t_repeat_location_crashes,t_highway_crashes
0,10001,Manhattan,2022,2182.0,3.0,0.0,18.0,528.0,110.0,0.0,450.5,17.0,17.0,18.0,2.0,1.0,1.0,0.0,0.0
1,10001,Manhattan,2023,1641.0,1.0,3.0,19.0,0.0,77.0,1.0,NaN,26.0,26.0,24.0,9.0,2.0,2.0,2.0,0.0
2,10001,Manhattan,2024,1469.0,3.0,6.0,24.0,61.0,77.0,1.0,NaN,22.0,22.0,25.0,18.0,0.0,0.0,1.0,0.0
3,10001,Manhattan,2025,1372.0,3.0,6.0,24.0,656.0,54.0,2.0,NaN,18.0,18.0,17.0,3.0,1.0,1.0,3.0,0.0
4,10002,Manhattan,2022,1050.0,7.0,0.0,20.0,937.0,90.0,7.0,862.0,22.0,20.0,25.0,4.0,2.0,2.0,4.0,2.0


,borough,year,month,year_month,h_filings_total,h_new_building,h_demolition,h_alteration_co,h_proposed_units,h_sidewalk_shed,h_curb_cut,t_crashes,t_crashes_street,t_severe_injuries,t_non_severe_injuries,t_fatalities,t_fatal_crashes,t_repeat_location_crashes,t_highway_crashes
0,Bronx,2022,1,2022-01,825,41,0,34,3138.0,78,11,42,38,40.0,8.0,5.0,5,3,4
1,Bronx,2022,2,2022-02,942,48,0,26,1993.0,70,16,43,33,40.0,14.0,6.0,5,2,10
2,Bronx,2022,3,2022-03,966,45,0,33,1842.0,139,13,42,39,43.0,4.0,3.0,3,2,3
3,Bronx,2022,4,2022-04,799,42,0,28,1321.0,96,9,32,26,30.0,2.0,2.0,2,2,6
4,Bronx,2022,5,2022-05,1018,28,0,25,845.0,113,8,48,43,46.0,8.0,8.0,8,3,5


,dimension,group,total_crashes,original_count,original_pct,spatial_join_count,spatial_join_pct,missing_count,missing_pct,fatal_crashes,fatal_crashes_missing_zip,fatal_crashes_missing_zip_pct,housing_rows_dropped_zip_year,crash_rows_dropped_zip_year,unverified_zip_10000_crashes
0,overall,all,13309,8072,60.65,4057,30.48,1180,8.87,1046,354,33.84,517.0,1147.0,44.0
1,year,2022,2879,469,16.29,2107,73.19,303,10.52,245,39,15.92,NaN,NaN,NaN
2,year,2023,3016,2055,68.14,650,21.55,311,10.31,237,105,44.30,NaN,NaN,NaN
3,year,2024,3116,2180,69.96,555,17.81,381,12.23,232,93,40.09,NaN,NaN,NaN
4,year,2025,2947,2367,80.32,464,15.74,116,3.94,211,64,30.33,NaN,NaN,NaN
